# Demo Prediksi Kultivar Wine
Notebook ini berisi demonstrasi sederhana menggunakan **Gradio** untuk memprediksi kultivar anggur berdasarkan beberapa fitur kimia (`Alcohol`, `Flavanoids`, `Color Intensity`, dan `Proline`).

**Cara pakai:** Jalankan sel kode di bawah, buka link `gradio.live` yang muncul, geser *slider* sesuai keinginan, lalu klik **Submit**.

In [1]:
!pip install -q gradio

import gradio as gr
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier

# 1. Ambil data & latih model
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(
    StandardScaler(),
    RandomForestClassifier(n_estimators=100, random_state=42)
)
model.fit(X_train, y_train)

# 2. Fungsi untuk memprediksi saat slider digeser
def prediksi(alcohol, flavanoids, color_intensity, proline):
    # Pakai nilai rata-rata untuk fitur lain supaya tidak error
    sampel = X_train.mean().copy()
    sampel["alcohol"] = alcohol
    sampel["flavanoids"] = flavanoids
    sampel["color_intensity"] = color_intensity
    sampel["proline"] = proline

    hasil = model.predict(pd.DataFrame([sampel]))[0]
    prob = model.predict_proba(pd.DataFrame([sampel]))[0]

    teks_hasil = f"Prediksi: Kultivar {hasil}"
    rincian_prob = {f"Kultivar {i}": float(prob[i]) for i in range(3)}
    return teks_hasil, rincian_prob

# 3. Buat tampilan antarmuka
rata2 = X_train.mean()

tampilan = gr.Interface(
    fn=prediksi,
    inputs=[
        gr.Slider(11.0, 15.0, value=round(rata2["alcohol"], 2), label="Alcohol"),
        gr.Slider(0.3, 5.2, value=round(rata2["flavanoids"], 2), label="Flavanoids"),
        gr.Slider(1.2, 13.0, value=round(rata2["color_intensity"], 2), label="Color Intensity"),
        gr.Slider(270.0, 1700.0, value=round(rata2["proline"], 1), label="Proline"),
    ],
    outputs=[
        gr.Textbox(label="Hasil Klasifikasi"),
        gr.Label(num_top_classes=3, label="Probabilitas Tiap Kultivar"),
    ],
    title="Demo Prediksi Kultivar Wine",
    description="Geser nilai fitur di kiri lalu klik Submit untuk melihat hasil prediksi model."
)

tampilan.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b61eb3bb2360171a76.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
